# W03-0 · Counting words and pairs in Python

In class we build a program that guesses the next word, like the suggestions above your phone keyboard. It works by
**counting**: which words appear, and which word comes right after which. This notebook covers the Python for that,
one small idea at a time: run each demo, guess the output before you look, then do the short ✍️ exercise. Replace `...`
with your code; each exercise checks itself.

We use four tiny sentences, short enough to check every answer by eye. They are the same four as on the slides.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shripada/ads6703-nlp/blob/main/w03/W03-0-counting-pairs.ipynb) · On Colab, run the next cell first. On your laptop, you can skip it.

In [ ]:
# Only for Google Colab: fetch the course folder (with the data) and move into this week's folder.
# On your laptop this cell does nothing.
import os, subprocess, sys
if "google.colab" in sys.modules:
    if os.path.exists("/content/ADS6703"):
        subprocess.run(["git", "-C", "/content/ADS6703", "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "https://github.com/shripada/ads6703-nlp.git", "/content/ADS6703"], check=True)
    os.chdir("/content/ADS6703/w03")

In [ ]:
# Four tiny sentences, already lower-case and without punctuation.
sentences = [
    "i like tea",
    "i like coffee",
    "you like tea",
    "i drink tea",
]
words = sentences[0].split()   # .split() cuts a sentence into a list of words at the spaces
words

## 1 · Counting with `Counter`

A `Counter` counts things for you: give it a list, and it tells you how many times each item appears.

| | |
|---|---|
| `c = Counter(xs)` | count every item in the list `xs` |
| `c["tea"]` | how many times `"tea"` appeared (0 if never: no error) |
| `c.most_common(3)` | the 3 most frequent items, with their counts |
| `sum(c.values())` | the total number of items counted |

In [ ]:
from collections import Counter

all_words = []
for s in sentences:
    all_words = all_words + s.split()   # add the words of this sentence to the end of the list
print(all_words)

counts = Counter(all_words)
print(counts)
print(counts["tea"], counts["milk"])    # 3, and 0 for a word that never appears
print(sum(counts.values()), "words in total")

### ✍️ Exercise 1: the share of one word

What share of all the words is `"like"`? Make `share_like` = (count of `"like"`) ÷ (total number of words).

In [ ]:
share_like = ...   # ✍️ the count of "like", divided by the total
print(share_like)
assert share_like == 3 / 12, "not quite: counts['like'] divided by sum(counts.values())"
print("✅")

## 2 · Neighbours: pairs of words with `zip`

To guess the *next* word we need every pair of neighbours: each word together with the word right after it.

`zip(a, b)` walks along two lists at the same time and gives you one item from each. If `b` is the same list moved on
by one place, `words[1:]`, you get every word with its right-hand neighbour.

In [ ]:
print(words)
print(words[1:])                    # the same list without its first word
print(list(zip(words, words[1:])))  # line them up: each word with the next one

Each pair is a **tuple**: two values in round brackets, like `("i", "like")`.

A sentence also has a start and an end. We mark them with two made-up words: `<s>` for *start* and `</s>` for *end*.
Then the first word gets a neighbour too (`("<s>", "i")`: *"i"* starts a sentence), and so does the last one
(`("tea", "</s>")`: *"tea"* ends it).

### ✍️ Exercise 2: pairs with start and end marks

Make `pairs`: the list of neighbour pairs of `marked`, which is the last sentence with `<s>` and `</s>` added.

In [ ]:
marked = ["<s>"] + sentences[3].split() + ["</s>"]   # ['<s>', 'i', 'drink', 'tea', '</s>']
pairs = ...   # ✍️ hint: list(zip(<the list>, <the list moved on by one>))
print(pairs)
assert pairs == [("<s>", "i"), ("i", "drink"), ("drink", "tea"), ("tea", "</s>")], "not quite"
print("✅ 3 words give 4 pairs: one more than the number of words.")

## 3 · Counting pairs

A `Counter` can count pairs just as easily as words. Count the pairs in all four sentences:

In [ ]:
pair_counts = Counter()
for s in sentences:
    marked = ["<s>"] + s.split() + ["</s>"]
    pair_counts.update(zip(marked, marked[1:]))   # .update adds these pairs to the counts

for pair, n in pair_counts.items():
    print(n, pair)

### ✍️ Exercise 3: read the pair counts

How many times does `"like"` come right after `"i"`? And `"drink"` right after `"you"`?

In [ ]:
i_like = ...      # ✍️ look up the pair ("i", "like")
you_drink = ...   # ✍️
print(i_like, you_drink)
assert (i_like, you_drink) == (2, 0), "not quite: look the pair up in pair_counts"
print("✅ 'you drink tea' is a fine sentence, but this pair was never seen.")

## 4 · What comes after a word?

For guessing, we want all the words that came after one particular word, with their counts. Go through every pair,
and keep the ones whose first word is the one we want.

### ✍️ Exercise 4: `next_counts(word)`

Return a dict: each word seen right after `word` → how many times. The loop is written for you; fill in the `if`.

In [ ]:
def next_counts(word):
    result = {}
    for (w1, w2), n in pair_counts.items():   # each pair, split into its two words, and its count
        if ...:              # ✍️ keep this pair only if its first word is `word`
            result[w2] = n
    return result
print(next_counts("i"))
print(next_counts("like"))
print(next_counts("<s>"))    # what starts a sentence
assert next_counts("i") == {"like": 2, "drink": 1}, "not quite"
print("✅")

## 5 · From counts to fractions

After *"i"* we saw *"like"* 2 times and *"drink"* 1 time. So the next word was *"like"* 2 times out of 3. That
fraction, 2/3, is the **probability** of *"like"* coming next after *"i"*, in these four sentences.

### ✍️ Exercise 5: a next-word probability

Make `p_like`: the count of `"like"` in `next_counts("i")`, divided by the total of all the counts in it.

In [ ]:
after = next_counts("i")
p_like = ...   # ✍️ the count of "like", divided by the total of all the counts in `after`
print(p_like)
assert p_like == 2 / 3, "not quite: 2 out of 3"
print("✅")

## 6 · Picking at random, with weights

To let a program *write* sentences, we pick the next word at random, but not fairly: a word seen twice as often should
be picked twice as often. `random.choices(items, weights=...)` does that. It returns a list, so take `[0]`.

Run the cell a few times: after *"i"* you should get *"like"* about 2 times in 3.

In [ ]:
import random

after = next_counts("i")          # {'like': 2, 'drink': 1}
picks = []
for _ in range(300):              # pick 300 times
    picks.append(random.choices(list(after), weights=list(after.values()))[0])
print(Counter(picks))             # about 200 'like' and 100 'drink'

## Cheat sheet

| | |
|---|---|
| `sentence.split()` | cut a sentence into words |
| `Counter(xs)` | count every item in a list |
| `c[x]` | the count of `x` (0 if never seen) |
| `sum(c.values())` | the total of all counts |
| `zip(xs, xs[1:])` | each item with the item after it |
| `["<s>"] + words + ["</s>"]` | mark the start and end of a sentence |
| `pair_counts.update(pairs)` | add a list of pairs to a Counter |
| `for (w1, w2), n in pair_counts.items():` | go through every pair and its count |
| `random.choices(items, weights=w)[0]` | one item, picked at random in proportion to `w` |